<a href="https://colab.research.google.com/github/vmyel/thesis_ref/blob/main/pd_main_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pandas

In [ ]:
pip install openpyxl

In [ ]:
pip install scikit-learn

In [ ]:
pip install numpy

In [ ]:
pip install matplotlib

In [ ]:
pip install seaborn

In [ ]:
# ============================================================
# 0.  Mount Google Drive & install/import libraries
# ============================================================
import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# 1.  USER-DEFINED PATHS  –– adjust to your Drive layout
# ============================================================
METADATA_PATH = 'PaHaW/PaHaW/PaHaW_files/corpus_PaHaW.xlsx'
SVC_ROOT = 'PaHaW/PaHaW/PaHaW_public/'

# METADATA_PATH = '/content/drive/MyDrive/teasis/PaHaW/PaHaW_files/corpus_PaHaW.xlsx'
# SVC_ROOT = '/content/drive/MyDrive/teasis/PaHaW/PaHaW_public/'
# ============================================================
# 2.  Load & inspect metadata
# ============================================================
meta = pd.read_excel(METADATA_PATH, dtype={'ID': str})

# Normalise column names (strip whitespace, consistent case)
meta.columns = meta.columns.str.strip()

# Make sure the subject ID column is zero-padded to 5 chars
# (adjust 'ID' to whatever the actual column header is)
meta['ID'] = meta['ID'].astype(str).str.zfill(5)

print("Metadata shape :", meta.shape)
print("Columns        :", meta.columns.tolist())

# ============================================================
# 3.  Filter out SEVERE stages  (H&Y / UPDRS V  >= 4.0)
#     NaN means Healthy Control → must be preserved
# ============================================================
before = len(meta)

# Keep row if UPDRS V is NaN (HC) OR score is below 4.0
meta_filtered = meta[meta['UPDRS V'].isna() | (meta['UPDRS V'] < 4.0)].copy()

after = len(meta_filtered)

print(f"Removed {before - after} subject(s) with UPDRS V >= 4.0")
print(f"Remaining subjects: {after}")   # should now be ~72

# Quick sanity check
print("\nUPDRS V distribution after filter:")
print(meta_filtered['UPDRS V'].value_counts(dropna=False).sort_index())

# ============================================================
# 4.  Assign group labels
#       • Healthy Controls : UPDRS V is NaN  (no H&Y score)
#       • Early PD         : UPDRS V  1.0 – 2.0
#       • Moderate PD      : UPDRS V  2.5 – 3.0
# ============================================================
def assign_group(row):
    score = row['UPDRS V']

    # Healthy controls have no UPDRS V score → NaN
    if pd.isna(score):
        return 'Healthy Control'
    elif 1.0 <= score <= 2.0:
        return 'Early PD'
    elif 2.5 <= score <= 3.0:
        return 'Moderate PD'
    else:
        return 'Unknown'   # safety catch

meta_filtered['Group'] = meta_filtered.apply(assign_group, axis=1)

print("Group distribution:")
print(meta_filtered['Group'].value_counts())
print()
print(meta_filtered[['ID', 'Disease', 'UPDRS V', 'Group']].head(72).to_string(index=False))

# ============================================================
# 5.  Parse a single SVC file
# ============================================================
def parse_svc(filepath: str) -> pd.DataFrame:
    """
    SVC format
    ----------
    Line 1   : number of samples (integer)
    Lines 2+ : Y  X  timestamp  button_state  azimuth  altitude  pressure
    """
    with open(filepath, 'r') as fh:
        lines = fh.read().splitlines()

    n_samples = int(lines[0].strip())
    data_lines = lines[1: n_samples + 1]          # safety slice

    records = []
    for line in data_lines:
        parts = line.split()
        if len(parts) < 7:
            continue
        records.append({
            'y'            : float(parts[0]),
            'x'            : float(parts[1]),
            'timestamp'    : float(parts[2]),
            'button_state' : int(parts[3]),
            'azimuth'      : float(parts[4]),
            'altitude'     : float(parts[5]),
            'pressure'     : float(parts[6]),
        })

    df = pd.DataFrame(records)
    df['n_declared'] = n_samples
    return df


# ============================================================
# 6.  Load ALL SVC files  (task number only, no session)
#     Filename format:  {subjectID}_{task}_{repetition}
#     e.g.  00098_6_1  →  subject=00098, task=6
# ============================================================
all_records = []
valid_ids   = set(meta_filtered['ID'].tolist())

for subj_folder in sorted(Path(SVC_ROOT).iterdir()):
    if not subj_folder.is_dir():
        continue

    subj_id = subj_folder.name.zfill(5)   # e.g. '98' → '00098'

    if subj_id not in valid_ids:
        continue

    meta_row = meta_filtered[meta_filtered['ID'] == subj_id].iloc[0]

    svc_files = sorted([f for f in subj_folder.iterdir() if f.is_file()])

    for svc_path in svc_files:
        try:
            svc_df = parse_svc(str(svc_path))
        except Exception as e:
            print(f"  [WARN] Could not parse {svc_path.name}: {e}")
            continue

        # e.g. '00098_6_1' → parts = ['00098', '6', '1']
        parts = svc_path.stem.split('_')
        task  = parts[1] if len(parts) > 1 else 'unknown'
        # parts[2] is just the repetition suffix ('1'), not used

        svc_df['subject_id'] = subj_id
        svc_df['task']       = task          # task number (1-8)
        svc_df['file_name']  = svc_path.name

        # Attach metadata
        svc_df['group']      = meta_row['Group']
        svc_df['updrs_v']    = meta_row['UPDRS V']
        svc_df['disease']    = meta_row['Disease']
        svc_df['age']        = meta_row['Age']
        svc_df['sex']        = meta_row['Sex']

        all_records.append(svc_df)

full_df = pd.concat(all_records, ignore_index=True)

print(f"Total stroke-point rows : {len(full_df):,}")
print(f"Unique subjects         : {full_df['subject_id'].nunique()}")
print(f"Unique SVC files        : {full_df['file_name'].nunique()}")
print(f"\nTasks found             : {sorted(full_df['task'].unique())}")

In [ ]:
# ============================================================
# 4.2  Patient-Level Stratified 5-Fold Cross-Validation Setup
# ============================================================
from sklearn.model_selection import StratifiedGroupKFold

# --- Define two classification objectives ---

# Objective 1: Detection (Healthy Control vs PD)
def assign_detection_label(group):
    if group == 'Healthy Control':
        return 0
    else:
        return 1  # Early PD or Moderate PD

# Objective 2: Staging (Early PD vs Moderate PD) — only PD subjects
def assign_staging_label(group):
    if group == 'Early PD':
        return 0
    elif group == 'Moderate PD':
        return 1
    else:
        return None  # Healthy controls excluded

# --- Build a subject-level dataframe for fold assignment ---
subject_df = meta_filtered[['ID', 'Group']].copy()
subject_df['detection_label'] = subject_df['Group'].apply(assign_detection_label)
subject_df['staging_label'] = subject_df['Group'].apply(assign_staging_label)

print("=" * 60)
print("DETECTION TASK — Label Distribution (subject-level)")
print("=" * 60)
print(subject_df['detection_label'].value_counts().rename({0: 'Healthy (0)', 1: 'PD (1)'}))
print()

staging_subject_df = subject_df.dropna(subset=['staging_label']).copy()
staging_subject_df['staging_label'] = staging_subject_df['staging_label'].astype(int)
print("=" * 60)
print("STAGING TASK — Label Distribution (subject-level)")
print("=" * 60)
print(staging_subject_df['staging_label'].value_counts().rename({0: 'Early PD (0)', 1: 'Moderate PD (1)'}))
print()

# --- Create the 5-Fold splitter ---
N_SPLITS = 5
RANDOM_STATE = 42

sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# --- Build file-level dataframes for each task ---
# Each SVC file is one sample; the group key is subject_id

file_level_df = full_df.groupby('file_name').first().reset_index()[
    ['file_name', 'subject_id', 'task', 'group', 'updrs_v', 'disease']
]
file_level_df['detection_label'] = file_level_df['group'].apply(assign_detection_label)
file_level_df['staging_label'] = file_level_df['group'].apply(assign_staging_label)

# Detection file-level
detect_file_df = file_level_df.copy()

# Staging file-level (PD only)
stage_file_df = file_level_df.dropna(subset=['staging_label']).copy()
stage_file_df['staging_label'] = stage_file_df['staging_label'].astype(int)

print(f"Detection samples (files): {len(detect_file_df)}")
print(f"Staging samples (files)  : {len(stage_file_df)}")
print()

# --- Generate and display fold splits ---
def generate_folds(df, label_col, task_name):
    """Generate fold indices and print summary."""
    X = df.index.values
    y = df[label_col].values
    groups = df['subject_id'].values

    folds = []
    print(f"{'='*60}")
    print(f"Fold Summary for: {task_name}")
    print(f"{'='*60}")
    for fold_idx, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups)):
        train_subjects = set(df.iloc[train_idx]['subject_id'].unique())
        test_subjects = set(df.iloc[test_idx]['subject_id'].unique())
        overlap = train_subjects & test_subjects

        print(f"\nFold {fold_idx + 1}:")
        print(f"  Train: {len(train_idx)} files, {len(train_subjects)} subjects")
        print(f"  Test : {len(test_idx)} files, {len(test_subjects)} subjects")
        print(f"  Train label dist: {dict(pd.Series(y[train_idx]).value_counts().sort_index())}")
        print(f"  Test  label dist: {dict(pd.Series(y[test_idx]).value_counts().sort_index())}")
        print(f"  Patient leakage : {'NONE ✓' if len(overlap) == 0 else f'WARNING: {overlap}'}")

        folds.append((train_idx, test_idx))
    return folds

detection_folds = generate_folds(detect_file_df, 'detection_label', 'DETECTION')
staging_folds = generate_folds(stage_file_df, 'staging_label', 'STAGING')

In [3]:
# ============================================================
# 4.3  Data Preprocessing
# ============================================================
from sklearn.preprocessing import StandardScaler
from scipy.stats import median_abs_deviation

# ============================================================
# 4.3 PATH A — Deep Learning Preprocessing
# ============================================================

SEQUENCE_LENGTH = 500  # L = 500 time-steps
DL_FEATURES = ['x', 'y', 'pressure', 'button_state']  # button_state = in-air status

def extract_dl_sequence(file_df):
    """
    Extract a raw time-series matrix for a single SVC file.
    Returns shape (n_timesteps, n_features).
    """
    seq = file_df[DL_FEATURES].values.astype(np.float32)
    return seq

def pad_or_clip_sequence(seq, target_length=SEQUENCE_LENGTH):
    """
    Clip if longer than target_length, zero-pad if shorter.
    Returns shape (target_length, n_features).
    """
    n_timesteps, n_features = seq.shape
    if n_timesteps >= target_length:
        return seq[:target_length]
    else:
        padded = np.zeros((target_length, n_features), dtype=np.float32)
        padded[:n_timesteps] = seq
        return padded

def prepare_dl_data(full_df, file_df, train_idx, test_idx, label_col):
    """
    Full Path A pipeline for a single fold.
    Returns X_train, X_test, y_train, y_test as numpy arrays.
    """
    train_files = file_df.iloc[train_idx]['file_name'].values
    test_files = file_df.iloc[test_idx]['file_name'].values

    # Step 1: Extract raw sequences
    train_sequences = []
    test_sequences = []

    for fname in train_files:
        fdata = full_df[full_df['file_name'] == fname]
        seq = extract_dl_sequence(fdata)
        train_sequences.append(seq)

    for fname in test_files:
        fdata = full_df[full_df['file_name'] == fname]
        seq = extract_dl_sequence(fdata)
        test_sequences.append(seq)

    # Step 2: Fit Z-score scaler on TRAINING data only
    # Concatenate all training time steps to fit scaler
    all_train_points = np.vstack(train_sequences)
    scaler = StandardScaler()
    scaler.fit(all_train_points)

    # Step 3: Scale both train and test
    train_scaled = [scaler.transform(seq) for seq in train_sequences]
    test_scaled = [scaler.transform(seq) for seq in test_sequences]

    # Step 4: Pad/Clip to fixed length
    X_train = np.array([pad_or_clip_sequence(seq) for seq in train_scaled])
    X_test = np.array([pad_or_clip_sequence(seq) for seq in test_scaled])

    # Step 5: Labels
    y_train = file_df.iloc[train_idx][label_col].values.astype(np.int64)
    y_test = file_df.iloc[test_idx][label_col].values.astype(np.int64)

    print(f"  [DL] X_train: {X_train.shape}, X_test: {X_test.shape}")
    print(f"  [DL] y_train dist: {dict(zip(*np.unique(y_train, return_counts=True)))}")
    print(f"  [DL] y_test  dist: {dict(zip(*np.unique(y_test, return_counts=True)))}")

    return X_train, X_test, y_train, y_test

# ============================================================
# 4.3 PATH B — Machine Learning Feature Engineering
# ============================================================

def compute_kinematics(file_data):
    """
    Compute velocity, acceleration, jerk from x, y, timestamp.
    Returns the file_data with new kinematic columns.
    """
    df = file_data.sort_values('timestamp').copy()

    dt = df['timestamp'].diff().fillna(1e-6)
    dt = dt.replace(0, 1e-6)  # avoid division by zero

    dx = df['x'].diff().fillna(0)
    dy = df['y'].diff().fillna(0)

    # Velocity components
    df['vx'] = dx / dt
    df['vy'] = dy / dt
    df['velocity'] = np.sqrt(df['vx']**2 + df['vy']**2)

    # Acceleration
    dvx = df['vx'].diff().fillna(0)
    dvy = df['vy'].diff().fillna(0)
    df['ax'] = dvx / dt
    df['ay'] = dvy / dt
    df['acceleration'] = np.sqrt(df['ax']**2 + df['ay']**2)

    # Jerk
    dax = df['ax'].diff().fillna(0)
    day = df['ay'].diff().fillna(0)
    df['jx'] = dax / dt
    df['jy'] = day / dt
    df['jerk'] = np.sqrt(df['jx']**2 + df['jy']**2)

    return df

def engineer_features(file_data):
    """
    Transform a single SVC file's time-series into a fixed-length feature vector.
    """
    df = compute_kinematics(file_data)

    features = {}

    # --- Statistical functionals for kinematic features ---
    for col in ['velocity', 'acceleration', 'jerk', 'pressure']:
        vals = df[col].replace([np.inf, -np.inf], np.nan).dropna()
        if len(vals) == 0:
            vals = pd.Series([0.0])

        features[f'{col}_mean'] = vals.mean()
        features[f'{col}_median'] = vals.median()
        features[f'{col}_std'] = vals.std()
        features[f'{col}_max'] = vals.max()
        features[f'{col}_min'] = vals.min()
        features[f'{col}_range'] = vals.max() - vals.min()
        features[f'{col}_25pct'] = vals.quantile(0.25)
        features[f'{col}_75pct'] = vals.quantile(0.75)
        features[f'{col}_iqr'] = vals.quantile(0.75) - vals.quantile(0.25)

    # --- Spatio-temporal / motor hesitation metrics ---
    total_time = df['timestamp'].max() - df['timestamp'].min()
    features['total_duration'] = total_time if total_time > 0 else 1e-6

    # In-air vs on-surface (button_state: 0 = in-air, 1 = on-surface)
    in_air_mask = df['button_state'] == 0
    on_surface_mask = df['button_state'] == 1

    in_air_time = df.loc[in_air_mask, 'timestamp'].diff().fillna(0).sum()
    on_surface_time = df.loc[on_surface_mask, 'timestamp'].diff().fillna(0).sum()

    features['in_air_time'] = in_air_time
    features['on_surface_time'] = on_surface_time
    features['in_air_ratio'] = in_air_time / features['total_duration']
    features['on_surface_ratio'] = on_surface_time / features['total_duration']

    # Number of in-air points and on-surface points
    features['n_in_air_points'] = in_air_mask.sum()
    features['n_on_surface_points'] = on_surface_mask.sum()

    # Stroke count: transitions from in-air to on-surface
    transitions = df['button_state'].diff().fillna(0)
    features['stroke_count'] = (transitions == 1).sum()

    # On-surface pressure stats
    on_surface_pressure = df.loc[on_surface_mask, 'pressure']
    if len(on_surface_pressure) > 0:
        features['on_surface_pressure_mean'] = on_surface_pressure.mean()
        features['on_surface_pressure_std'] = on_surface_pressure.std()
    else:
        features['on_surface_pressure_mean'] = 0.0
        features['on_surface_pressure_std'] = 0.0

    # Total path length
    dx = df['x'].diff().fillna(0)
    dy = df['y'].diff().fillna(0)
    features['total_path_length'] = np.sqrt(dx**2 + dy**2).sum()

    # Number of data points
    features['n_points'] = len(df)

    return features

def prepare_ml_data(full_df, file_df, train_idx, test_idx, label_col):
    """
    Full Path B pipeline for a single fold.
    Returns X_train, X_test, y_train, y_test as numpy arrays/DataFrames.
    """
    train_files = file_df.iloc[train_idx]['file_name'].values
    test_files = file_df.iloc[test_idx]['file_name'].values

    # Step 1: Engineer features for each file
    train_features = []
    for fname in train_files:
        fdata = full_df[full_df['file_name'] == fname]
        feat = engineer_features(fdata)
        feat['file_name'] = fname
        train_features.append(feat)

    test_features = []
    for fname in test_files:
        fdata = full_df[full_df['file_name'] == fname]
        feat = engineer_features(fdata)
        feat['file_name'] = fname
        test_features.append(feat)

    train_feat_df = pd.DataFrame(train_features)
    test_feat_df = pd.DataFrame(test_features)

    # Feature column names (exclude file_name)
    feature_cols = [c for c in train_feat_df.columns if c != 'file_name']

    # Replace inf with NaN, then fill NaN with 0
    train_feat_df[feature_cols] = train_feat_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    test_feat_df[feature_cols] = test_feat_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

    # Step 2: Z-score normalization (fit on train only)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_feat_df[feature_cols])
    X_test = scaler.transform(test_feat_df[feature_cols])

    # Step 3: Labels
    y_train = file_df.iloc[train_idx][label_col].values.astype(np.int64)
    y_test = file_df.iloc[test_idx][label_col].values.astype(np.int64)

    print(f"  [ML] X_train: {X_train.shape}, X_test: {X_test.shape}")
    print(f"  [ML] y_train dist: {dict(zip(*np.unique(y_train, return_counts=True)))}")
    print(f"  [ML] y_test  dist: {dict(zip(*np.unique(y_test, return_counts=True)))}")

    return X_train, X_test, y_train, y_test, feature_cols, scaler

In [ ]:
# ============================================================
# 4.4  Modeling Methodology (UPDATED with Hyperparameter Tuning)
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import ParameterGrid
import itertools
import time

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# --- Install xgboost if needed ---
try:
    import xgboost as xgb
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'xgboost'])
    import xgboost as xgb

# ============================================================
# HYPERPARAMETER SEARCH SPACES
# ============================================================

# DL hyperparameter grid (GRU & LSTM share the same space)
DL_PARAM_GRID = {
    'hidden_size': [64, 128, 256],
    'num_layers': [1, 2, 3],
    'dropout': [0.2, 0.3, 0.5],
    'lr': [1e-3, 5e-4, 1e-4],
    'batch_size': [16, 32, 64],
}

# For efficiency: use a reduced search (top combinations) rather than full grid
# Full grid = 3*3*3*3*3 = 243 combos — too expensive per fold
# We'll use a randomized subset
DL_N_RANDOM_CONFIGS = 12  # Number of random configs to try per fold

# Random Forest hyperparameter grid
RF_PARAM_GRID = {
    'n_estimators': [100, 300, 500],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
}
RF_N_RANDOM_CONFIGS = 20

# XGBoost hyperparameter grid
XGB_PARAM_GRID = {
    'n_estimators': [100, 300, 500],
    'max_depth': [3, 6, 9, 12],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
}
XGB_N_RANDOM_CONFIGS = 20


def sample_random_configs(param_grid, n_configs, random_state=42):
    """
    Sample n_configs random hyperparameter combinations from the grid.
    If total combos <= n_configs, return all of them.
    """
    rng = np.random.RandomState(random_state)
    all_combos = list(ParameterGrid(param_grid))
    if len(all_combos) <= n_configs:
        return all_combos
    indices = rng.choice(len(all_combos), size=n_configs, replace=False)
    return [all_combos[i] for i in indices]


# ============================================================
# PATH A: Deep Learning Models (GRU & LSTM) — with HP Tuning
# ============================================================

class GRUClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, num_classes=2, dropout=0.3):
        super(GRUClassifier, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.gru(x)
        out = out[:, -1, :]
        out = self.dropout(out)
        out = self.fc(out)
        return out


class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, num_classes=2, dropout=0.3):
        super(LSTMClassifier, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, (_, _) = self.lstm(x)
        out = out[:, -1, :]
        out = self.dropout(out)
        out = self.fc(out)
        return out


def train_dl_model(model, X_train, y_train, X_val, y_val,
                   epochs=100, batch_size=32, lr=1e-3, patience=15,
                   verbose=True):
    """
    Train a PyTorch DL model with class weighting and early stopping.
    Returns the trained model, training history, and best validation loss.
    """
    model = model.to(device)

    # Compute class weights
    classes = np.unique(y_train)
    cw = compute_class_weight('balanced', classes=classes, y=y_train)
    class_weights = torch.FloatTensor(cw).to(device)
    if verbose:
        print(f"    Class weights: {dict(zip(classes, cw.round(4)))}")

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

    # Create DataLoaders
    train_dataset = TensorDataset(
        torch.FloatTensor(X_train),
        torch.LongTensor(y_train)
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    X_val_tensor = torch.FloatTensor(X_val).to(device)
    y_val_tensor = torch.LongTensor(y_val).to(device)

    # Training loop with early stopping
    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        avg_train_loss = epoch_loss / len(train_loader)

        # Validation loss
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_tensor)
            val_loss = criterion(val_outputs, y_val_tensor).item()

        scheduler.step(val_loss)

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(val_loss)

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1

        if verbose and (epoch + 1) % 20 == 0:
            print(f"      Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss:.4f}")

        if patience_counter >= patience:
            if verbose:
                print(f"      Early stopping at epoch {epoch+1}")
            break

    # Restore best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return model, history, best_val_loss


def predict_dl_model(model, X_test):
    """Get predictions and probabilities from a DL model."""
    model.eval()
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X_test).to(device)
        outputs = model(X_tensor)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)
    return preds, probs


def tune_dl_model(model_class, input_size, num_classes,
                  X_train, y_train, X_val, y_val,
                  model_name, n_configs=DL_N_RANDOM_CONFIGS):
    """
    Hyperparameter tuning for DL models using random search.
    Uses validation loss as the selection criterion.
    Returns the best model, best params, and tuning log.
    """
    print(f"\n    >> Hyperparameter Tuning for {model_name} ({n_configs} configs)...")

    configs = sample_random_configs(DL_PARAM_GRID, n_configs)
    print(f"    Searching {len(configs)} hyperparameter configurations...")

    best_val_loss = float('inf')
    best_model = None
    best_params = None
    best_history = None
    tuning_log = []

    for cfg_idx, cfg in enumerate(configs):
        print(f"\n    Config {cfg_idx + 1}/{len(configs)}: {cfg}")
        start_time = time.time()

        try:
            model = model_class(
                input_size=input_size,
                hidden_size=cfg['hidden_size'],
                num_layers=cfg['num_layers'],
                num_classes=num_classes,
                dropout=cfg['dropout']
            )

            trained_model, history, val_loss = train_dl_model(
                model, X_train, y_train, X_val, y_val,
                epochs=100,
                batch_size=cfg['batch_size'],
                lr=cfg['lr'],
                patience=15,
                verbose=False  # Suppress per-epoch output during tuning
            )

            elapsed = time.time() - start_time

            # Also compute validation F1 for logging
            preds, probs = predict_dl_model(trained_model, X_val)
            val_f1 = f1_score(y_val, preds, zero_division=0)
            val_acc = np.mean(preds == y_val)

            log_entry = {**cfg, 'val_loss': val_loss, 'val_f1': val_f1,
                         'val_accuracy': val_acc, 'time_sec': elapsed}
            tuning_log.append(log_entry)

            print(f"      Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f} | "
                  f"Val Acc: {val_acc:.4f} | Time: {elapsed:.1f}s")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model = trained_model
                best_params = cfg
                best_history = history

        except Exception as e:
            print(f"      [FAILED] {e}")
            tuning_log.append({**cfg, 'val_loss': float('inf'), 'val_f1': 0.0,
                               'val_accuracy': 0.0, 'time_sec': 0, 'error': str(e)})

    print(f"\n    ✓ Best {model_name} Config: {best_params}")
    print(f"      Best Val Loss: {best_val_loss:.4f}")

    tuning_log_df = pd.DataFrame(tuning_log).sort_values('val_loss')
    return best_model, best_params, best_history, tuning_log_df


# ============================================================
# PATH B: Machine Learning Models (RF & XGBoost) — with HP Tuning
# ============================================================

def tune_rf_model(X_train, y_train, X_val, y_val, n_configs=RF_N_RANDOM_CONFIGS):
    """
    Hyperparameter tuning for Random Forest using random search.
    Uses validation F1 as the selection criterion.
    """
    print(f"\n    >> Hyperparameter Tuning for Random Forest ({n_configs} configs)...")

    configs = sample_random_configs(RF_PARAM_GRID, n_configs)
    print(f"    Searching {len(configs)} hyperparameter configurations...")

    best_val_f1 = -1
    best_model = None
    best_params = None
    tuning_log = []

    for cfg_idx, cfg in enumerate(configs):
        start_time = time.time()

        try:
            rf = RandomForestClassifier(
                n_estimators=cfg['n_estimators'],
                max_depth=cfg['max_depth'],
                min_samples_split=cfg['min_samples_split'],
                min_samples_leaf=cfg['min_samples_leaf'],
                max_features=cfg['max_features'],
                class_weight='balanced',
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
            rf.fit(X_train, y_train)

            val_preds = rf.predict(X_val)
            val_f1 = f1_score(y_val, val_preds, zero_division=0)
            val_acc = np.mean(val_preds == y_val)

            elapsed = time.time() - start_time

            log_entry = {**cfg, 'val_f1': val_f1, 'val_accuracy': val_acc, 'time_sec': elapsed}
            tuning_log.append(log_entry)

            if (cfg_idx + 1) % 5 == 0 or cfg_idx == 0:
                print(f"    Config {cfg_idx + 1}/{len(configs)}: F1={val_f1:.4f}, "
                      f"Acc={val_acc:.4f} | {elapsed:.1f}s")

            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_model = rf
                best_params = cfg

        except Exception as e:
            print(f"    Config {cfg_idx + 1} [FAILED]: {e}")
            tuning_log.append({**cfg, 'val_f1': 0.0, 'val_accuracy': 0.0,
                               'time_sec': 0, 'error': str(e)})

    print(f"\n    ✓ Best RF Config: {best_params}")
    print(f"      Best Val F1: {best_val_f1:.4f}")

    tuning_log_df = pd.DataFrame(tuning_log).sort_values('val_f1', ascending=False)
    return best_model, best_params, tuning_log_df


def tune_xgb_model(X_train, y_train, X_val, y_val, n_configs=XGB_N_RANDOM_CONFIGS):
    """
    Hyperparameter tuning for XGBoost using random search.
    Uses validation F1 as the selection criterion.
    """
    print(f"\n    >> Hyperparameter Tuning for XGBoost ({n_configs} configs)...")

    configs = sample_random_configs(XGB_PARAM_GRID, n_configs)
    print(f"    Searching {len(configs)} hyperparameter configurations...")

    # Compute scale_pos_weight
    classes, counts = np.unique(y_train, return_counts=True)
    if len(classes) == 2:
        scale_pos_weight = counts[0] / counts[1]
    else:
        scale_pos_weight = 1.0

    best_val_f1 = -1
    best_model = None
    best_params = None
    tuning_log = []

    for cfg_idx, cfg in enumerate(configs):
        start_time = time.time()

        try:
            xgb_model = xgb.XGBClassifier(
                n_estimators=cfg['n_estimators'],
                max_depth=cfg['max_depth'],
                learning_rate=cfg['learning_rate'],
                subsample=cfg['subsample'],
                colsample_bytree=cfg['colsample_bytree'],
                min_child_weight=cfg['min_child_weight'],
                scale_pos_weight=scale_pos_weight,
                eval_metric='logloss',
                use_label_encoder=False,
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
            xgb_model.fit(X_train, y_train, verbose=False)

            val_preds = xgb_model.predict(X_val)
            val_f1 = f1_score(y_val, val_preds, zero_division=0)
            val_acc = np.mean(val_preds == y_val)

            elapsed = time.time() - start_time

            log_entry = {**cfg, 'val_f1': val_f1, 'val_accuracy': val_acc, 'time_sec': elapsed}
            tuning_log.append(log_entry)

            if (cfg_idx + 1) % 5 == 0 or cfg_idx == 0:
                print(f"    Config {cfg_idx + 1}/{len(configs)}: F1={val_f1:.4f}, "
                      f"Acc={val_acc:.4f} | {elapsed:.1f}s")

            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_model = xgb_model
                best_params = cfg

        except Exception as e:
            print(f"    Config {cfg_idx + 1} [FAILED]: {e}")
            tuning_log.append({**cfg, 'val_f1': 0.0, 'val_accuracy': 0.0,
                               'time_sec': 0, 'error': str(e)})

    print(f"\n    ✓ Best XGBoost Config: {best_params}")
    print(f"      Best Val F1: {best_val_f1:.4f}")

    tuning_log_df = pd.DataFrame(tuning_log).sort_values('val_f1', ascending=False)
    return best_model, best_params, tuning_log_df


def predict_ml_model(model, X_test):
    """Get predictions and probabilities from an ML model."""
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)
    return preds, probs


# ============================================================
# 4.5  Performance Metrics (UPDATED — accuracy always included)
# ============================================================
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score, classification_report, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt


def compute_fold_metrics(y_true, y_pred, y_prob, num_classes=2):
    """
    Compute all metrics for a single fold.
    Returns a dictionary of metric values.
    """
    metrics = {}

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    metrics['confusion_matrix'] = cm

    # Accuracy — always computed regardless of num_classes
    metrics['accuracy'] = accuracy_score(y_true, y_pred)

    if num_classes == 2:
        tn, fp, fn, tp = cm.ravel()
        metrics['TP'] = tp
        metrics['TN'] = tn
        metrics['FP'] = fp
        metrics['FN'] = fn

        # Precision, Recall, F1
        metrics['precision'] = precision_score(y_true, y_pred, zero_division=0)
        metrics['recall'] = recall_score(y_true, y_pred, zero_division=0)
        metrics['f1_score'] = f1_score(y_true, y_pred, zero_division=0)

        # Specificity
        metrics['specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        # AUC-ROC
        if y_prob is not None and len(np.unique(y_true)) > 1:
            metrics['auc_roc'] = roc_auc_score(y_true, y_prob[:, 1])
        else:
            metrics['auc_roc'] = float('nan')
    else:
        # Macro averages for multi-class
        metrics['precision'] = precision_score(y_true, y_pred, average='macro', zero_division=0)
        metrics['recall'] = recall_score(y_true, y_pred, average='macro', zero_division=0)
        metrics['f1_score'] = f1_score(y_true, y_pred, average='macro', zero_division=0)
        if y_prob is not None and len(np.unique(y_true)) > 1:
            metrics['auc_roc'] = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
        else:
            metrics['auc_roc'] = float('nan')

    return metrics


def print_fold_metrics(metrics, fold_idx, model_name):
    """Pretty-print metrics for a single fold."""
    print(f"\n  --- {model_name} | Fold {fold_idx + 1} ---")
    print(f"  Accuracy  : {metrics['accuracy']:.4f}")
    print(f"  Precision : {metrics['precision']:.4f}")
    print(f"  Recall    : {metrics['recall']:.4f}")
    print(f"  F1-Score  : {metrics['f1_score']:.4f}")
    print(f"  AUC-ROC   : {metrics['auc_roc']:.4f}")
    if 'specificity' in metrics:
        print(f"  Specificity: {metrics['specificity']:.4f}")
    print(f"  Confusion Matrix:\n{metrics['confusion_matrix']}")


def summarize_cv_results(all_fold_metrics, model_name, task_name):
    """
    Summarize metrics across K folds: report mean ± std.
    Returns a summary dict.
    """
    metric_keys = ['accuracy', 'precision', 'recall', 'f1_score', 'auc_roc']
    if 'specificity' in all_fold_metrics[0]:
        metric_keys.append('specificity')

    summary = {}
    print(f"\n{'='*60}")
    print(f"  {model_name} | {task_name} | 5-Fold CV Summary")
    print(f"{'='*60}")
    for key in metric_keys:
        values = [m[key] for m in all_fold_metrics if not np.isnan(m.get(key, float('nan')))]
        mean_val = np.mean(values) if values else float('nan')
        std_val = np.std(values) if values else float('nan')
        summary[f'{key}_mean'] = mean_val
        summary[f'{key}_std'] = std_val
        print(f"  {key:12s}: {mean_val:.4f} ± {std_val:.4f}")

    return summary


def plot_confusion_matrices(all_fold_metrics, model_name, task_name, class_names):
    """Plot confusion matrices for all folds."""
    fig, axes = plt.subplots(1, N_SPLITS, figsize=(4 * N_SPLITS, 4))
    for i, metrics in enumerate(all_fold_metrics):
        disp = ConfusionMatrixDisplay(
            confusion_matrix=metrics['confusion_matrix'],
            display_labels=class_names
        )
        disp.plot(ax=axes[i], cmap='Blues', values_format='d')
        axes[i].set_title(f'Fold {i+1}')
    fig.suptitle(f'{model_name} — {task_name}', fontsize=14)
    plt.tight_layout()
    plt.show()


# ============================================================
# HELPER: Create a validation split from training data
# (for hyperparameter tuning within each CV fold)
# ============================================================
from sklearn.model_selection import StratifiedShuffleSplit


def create_tuning_val_split(X, y, groups=None, val_fraction=0.2, random_state=42):
    """
    Create a single stratified train/validation split from the training fold.
    Used internally for hyperparameter selection.
    Returns indices into the input arrays.
    """
    sss = StratifiedShuffleSplit(n_splits=1, test_size=val_fraction, random_state=random_state)
    train_sub_idx, val_sub_idx = next(sss.split(X, y))
    return train_sub_idx, val_sub_idx


# ============================================================
# MAIN EXPERIMENT LOOP (UPDATED with HP Tuning)
# ============================================================
import warnings
warnings.filterwarnings('ignore')

# Storage for all results
all_results = {}

# Storage for all tuning logs
all_tuning_logs = {}


def run_experiment(task_name, file_df, label_col, folds, class_names):
    """
    Run the complete experiment pipeline for a given task,
    including hyperparameter tuning within each fold.
    """
    task_results = {}
    task_tuning_logs = {}

    # Storage for each model
    model_fold_metrics = {
        'GRU': [], 'LSTM': [], 'RF': [], 'XGBoost': []
    }
    model_best_params = {
        'GRU': [], 'LSTM': [], 'RF': [], 'XGBoost': []
    }
    best_fold_info = {
        'GRU': {'best_f1': -1}, 'LSTM': {'best_f1': -1},
        'RF': {'best_f1': -1}, 'XGBoost': {'best_f1': -1}
    }

    input_size = len(DL_FEATURES)  # 4 features
    num_classes = len(class_names)

    for fold_idx, (train_idx, test_idx) in enumerate(folds):
        print(f"\n{'#'*60}")
        print(f"# {task_name} — FOLD {fold_idx + 1}/{N_SPLITS}")
        print(f"{'#'*60}")

        # ---- Path A: Deep Learning Preprocessing ----
        print("\n  >> Preparing DL data...")
        X_train_dl, X_test_dl, y_train_dl, y_test_dl = prepare_dl_data(
            full_df, file_df, train_idx, test_idx, label_col
        )

        # ---- Path B: ML Feature Engineering ----
        print("\n  >> Preparing ML data...")
        X_train_ml, X_test_ml, y_train_ml, y_test_ml, feat_cols, ml_scaler = prepare_ml_data(
            full_df, file_df, train_idx, test_idx, label_col
        )

        # ============================================================
        # Create internal validation split for HP tuning
        # (from within the training fold only — no test data leakage)
        # ============================================================

        # DL validation split
        dl_train_sub_idx, dl_val_sub_idx = create_tuning_val_split(
            X_train_dl, y_train_dl, val_fraction=0.2, random_state=RANDOM_STATE + fold_idx
        )
        X_tune_train_dl = X_train_dl[dl_train_sub_idx]
        y_tune_train_dl = y_train_dl[dl_train_sub_idx]
        X_tune_val_dl = X_train_dl[dl_val_sub_idx]
        y_tune_val_dl = y_train_dl[dl_val_sub_idx]

        print(f"\n  DL tuning split: train={len(dl_train_sub_idx)}, val={len(dl_val_sub_idx)}")

        # ML validation split
        ml_train_sub_idx, ml_val_sub_idx = create_tuning_val_split(
            X_train_ml, y_train_ml, val_fraction=0.2, random_state=RANDOM_STATE + fold_idx
        )
        X_tune_train_ml = X_train_ml[ml_train_sub_idx]
        y_tune_train_ml = y_train_ml[ml_train_sub_idx]
        X_tune_val_ml = X_train_ml[ml_val_sub_idx]
        y_tune_val_ml = y_train_ml[ml_val_sub_idx]

        print(f"  ML tuning split: train={len(ml_train_sub_idx)}, val={len(ml_val_sub_idx)}")

        # ======== GRU (with HP Tuning) ========
        print("\n  >> Tuning & Training GRU...")
        best_gru, best_gru_params, _, gru_tune_log = tune_dl_model(
            GRUClassifier, input_size, num_classes,
            X_tune_train_dl, y_tune_train_dl, X_tune_val_dl, y_tune_val_dl,
            model_name='GRU', n_configs=DL_N_RANDOM_CONFIGS
        )
        model_best_params['GRU'].append(best_gru_params)
        task_tuning_logs[f'GRU_fold{fold_idx+1}'] = gru_tune_log

        # Retrain on FULL training data with best params
        print(f"\n    >> Retraining GRU on full training fold with best params: {best_gru_params}")
        gru_model = GRUClassifier(
            input_size=input_size,
            hidden_size=best_gru_params['hidden_size'],
            num_layers=best_gru_params['num_layers'],
            num_classes=num_classes,
            dropout=best_gru_params['dropout']
        )
        gru_model, gru_history, _ = train_dl_model(
            gru_model, X_train_dl, y_train_dl, X_test_dl, y_test_dl,
            epochs=100,
            batch_size=best_gru_params['batch_size'],
            lr=best_gru_params['lr'],
            patience=15,
            verbose=True
        )
        gru_preds, gru_probs = predict_dl_model(gru_model, X_test_dl)
        gru_metrics = compute_fold_metrics(y_test_dl, gru_preds, gru_probs, num_classes)
        print_fold_metrics(gru_metrics, fold_idx, 'GRU')
        model_fold_metrics['GRU'].append(gru_metrics)

        if gru_metrics['f1_score'] > best_fold_info['GRU']['best_f1']:
            best_fold_info['GRU'] = {
                'best_f1': gru_metrics['f1_score'],
                'fold_idx': fold_idx,
                'model': gru_model,
                'X_test': X_test_dl, 'y_test': y_test_dl,
                'y_pred': gru_preds, 'y_prob': gru_probs,
                'history': gru_history,
                'best_params': best_gru_params
            }

        # ======== LSTM (with HP Tuning) ========
        print("\n  >> Tuning & Training LSTM...")
        best_lstm, best_lstm_params, _, lstm_tune_log = tune_dl_model(
            LSTMClassifier, input_size, num_classes,
            X_tune_train_dl, y_tune_train_dl, X_tune_val_dl, y_tune_val_dl,
            model_name='LSTM', n_configs=DL_N_RANDOM_CONFIGS
        )
        model_best_params['LSTM'].append(best_lstm_params)
        task_tuning_logs[f'LSTM_fold{fold_idx+1}'] = lstm_tune_log

        # Retrain on FULL training data with best params
        print(f"\n    >> Retraining LSTM on full training fold with best params: {best_lstm_params}")
        lstm_model = LSTMClassifier(
            input_size=input_size,
            hidden_size=best_lstm_params['hidden_size'],
            num_layers=best_lstm_params['num_layers'],
            num_classes=num_classes,
            dropout=best_lstm_params['dropout']
        )
        lstm_model, lstm_history, _ = train_dl_model(
            lstm_model, X_train_dl, y_train_dl, X_test_dl, y_test_dl,
            epochs=100,
            batch_size=best_lstm_params['batch_size'],
            lr=best_lstm_params['lr'],
            patience=15,
            verbose=True
        )
        lstm_preds, lstm_probs = predict_dl_model(lstm_model, X_test_dl)
        lstm_metrics = compute_fold_metrics(y_test_dl, lstm_preds, lstm_probs, num_classes)
        print_fold_metrics(lstm_metrics, fold_idx, 'LSTM')
        model_fold_metrics['LSTM'].append(lstm_metrics)

        if lstm_metrics['f1_score'] > best_fold_info['LSTM']['best_f1']:
            best_fold_info['LSTM'] = {
                'best_f1': lstm_metrics['f1_score'],
                'fold_idx': fold_idx,
                'model': lstm_model,
                'X_test': X_test_dl, 'y_test': y_test_dl,
                'y_pred': lstm_preds, 'y_prob': lstm_probs,
                'history': lstm_history,
                'best_params': best_lstm_params
            }

        # ======== Random Forest (with HP Tuning) ========
        print("\n  >> Tuning & Training Random Forest...")
        best_rf, best_rf_params, rf_tune_log = tune_rf_model(
            X_tune_train_ml, y_tune_train_ml, X_tune_val_ml, y_tune_val_ml,
            n_configs=RF_N_RANDOM_CONFIGS
        )
        model_best_params['RF'].append(best_rf_params)
        task_tuning_logs[f'RF_fold{fold_idx+1}'] = rf_tune_log

        # Retrain on FULL training data with best params
        print(f"\n    >> Retraining RF on full training fold with best params: {best_rf_params}")
        rf_model = RandomForestClassifier(
            n_estimators=best_rf_params['n_estimators'],
            max_depth=best_rf_params['max_depth'],
            min_samples_split=best_rf_params['min_samples_split'],
            min_samples_leaf=best_rf_params['min_samples_leaf'],
            max_features=best_rf_params['max_features'],
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
        rf_model.fit(X_train_ml, y_train_ml)
        rf_preds, rf_probs = predict_ml_model(rf_model, X_test_ml)
        rf_metrics = compute_fold_metrics(y_test_ml, rf_preds, rf_probs, num_classes)
        print_fold_metrics(rf_metrics, fold_idx, 'RF')
        model_fold_metrics['RF'].append(rf_metrics)

        if rf_metrics['f1_score'] > best_fold_info['RF']['best_f1']:
            best_fold_info['RF'] = {
                'best_f1': rf_metrics['f1_score'],
                'fold_idx': fold_idx,
                'model': rf_model,
                'X_train': X_train_ml, 'X_test': X_test_ml,
                'y_train': y_train_ml, 'y_test': y_test_ml,
                'y_pred': rf_preds, 'y_prob': rf_probs,
                'feature_names': feat_cols,
                'best_params': best_rf_params
            }

        # ======== XGBoost (with HP Tuning) ========
        print("\n  >> Tuning & Training XGBoost...")
        best_xgb, best_xgb_params, xgb_tune_log = tune_xgb_model(
            X_tune_train_ml, y_tune_train_ml, X_tune_val_ml, y_tune_val_ml,
            n_configs=XGB_N_RANDOM_CONFIGS
        )
        model_best_params['XGBoost'].append(best_xgb_params)
        task_tuning_logs[f'XGBoost_fold{fold_idx+1}'] = xgb_tune_log

        # Retrain on FULL training data with best params
        print(f"\n    >> Retraining XGBoost on full training fold with best params: {best_xgb_params}")
        classes, counts = np.unique(y_train_ml, return_counts=True)
        scale_pos_weight = counts[0] / counts[1] if len(classes) == 2 else 1.0

        xgb_model = xgb.XGBClassifier(
            n_estimators=best_xgb_params['n_estimators'],
            max_depth=best_xgb_params['max_depth'],
            learning_rate=best_xgb_params['learning_rate'],
            subsample=best_xgb_params['subsample'],
            colsample_bytree=best_xgb_params['colsample_bytree'],
            min_child_weight=best_xgb_params['min_child_weight'],
            scale_pos_weight=scale_pos_weight,
            eval_metric='logloss',
            use_label_encoder=False,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
        xgb_model.fit(X_train_ml, y_train_ml, verbose=False)
        xgb_preds, xgb_probs = predict_ml_model(xgb_model, X_test_ml)
        xgb_metrics = compute_fold_metrics(y_test_ml, xgb_preds, xgb_probs, num_classes)
        print_fold_metrics(xgb_metrics, fold_idx, 'XGBoost')
        model_fold_metrics['XGBoost'].append(xgb_metrics)

        if xgb_metrics['f1_score'] > best_fold_info['XGBoost']['best_f1']:
            best_fold_info['XGBoost'] = {
                'best_f1': xgb_metrics['f1_score'],
                'fold_idx': fold_idx,
                'model': xgb_model,
                'X_train': X_train_ml, 'X_test': X_test_ml,
                'y_train': y_train_ml, 'y_test': y_test_ml,
                'y_pred': xgb_preds, 'y_prob': xgb_probs,
                'feature_names': feat_cols,
                'best_params': best_xgb_params
            }

    # ---- Summarize all models ----
    print(f"\n\n{'*'*60}")
    print(f"* SUMMARY: {task_name}")
    print(f"{'*'*60}")

    summaries = {}
    for model_name in ['GRU', 'LSTM', 'RF', 'XGBoost']:
        summaries[model_name] = summarize_cv_results(
            model_fold_metrics[model_name], model_name, task_name
        )
        plot_confusion_matrices(
            model_fold_metrics[model_name], model_name, task_name, class_names
        )

    # ---- Print best hyperparameters per fold ----
    print(f"\n{'='*60}")
    print(f"  BEST HYPERPARAMETERS PER FOLD — {task_name}")
    print(f"{'='*60}")
    for model_name in ['GRU', 'LSTM', 'RF', 'XGBoost']:
        print(f"\n  {model_name}:")
        for fold_idx, params in enumerate(model_best_params[model_name]):
            print(f"    Fold {fold_idx + 1}: {params}")

    task_results['fold_metrics'] = model_fold_metrics
    task_results['summaries'] = summaries
    task_results['best_fold_info'] = best_fold_info
    task_results['best_params_per_fold'] = model_best_params
    task_results['tuning_logs'] = task_tuning_logs

    return task_results


# ============================================================
# RUN DETECTION TASK
# ============================================================
print("\n" + "=" * 70)
print("   RUNNING DETECTION TASK: Healthy Control vs PD")
print("=" * 70)

detection_results = run_experiment(
    task_name='Detection',
    file_df=detect_file_df,
    label_col='detection_label',
    folds=detection_folds,
    class_names=['Healthy', 'PD']
)

all_results['Detection'] = detection_results

# ============================================================
# RUN STAGING TASK
# ============================================================
print("\n" + "=" * 70)
print("   RUNNING STAGING TASK: Early PD vs Moderate PD")
print("=" * 70)

staging_results = run_experiment(
    task_name='Staging',
    file_df=stage_file_df,
    label_col='staging_label',
    folds=staging_folds,
    class_names=['Early PD', 'Moderate PD']
)

all_results['Staging'] = staging_results

# ============================================================
# 4.5  Final Results Comparison Table (UPDATED with accuracy)
# ============================================================

def create_results_table(all_results):
    """Create a comprehensive results comparison table including accuracy."""
    rows = []
    for task_name, task_data in all_results.items():
        for model_name, summary in task_data['summaries'].items():
            row = {
                'Task': task_name,
                'Model': model_name,
                'Path': 'DL' if model_name in ['GRU', 'LSTM'] else 'ML'
            }
            for metric in ['accuracy', 'precision', 'recall', 'f1_score', 'auc_roc']:
                mean = summary.get(f'{metric}_mean', float('nan'))
                std = summary.get(f'{metric}_std', float('nan'))
                row[f'{metric}'] = f"{mean:.4f} ± {std:.4f}"
            rows.append(row)

    results_df = pd.DataFrame(rows)
    print("\n" + "=" * 100)
    print("  COMPREHENSIVE RESULTS TABLE (with Hyperparameter Tuning)")
    print("=" * 100)
    print(results_df.to_string(index=False))
    return results_df

results_table = create_results_table(all_results)

# --- ROC Curves ---
from sklearn.metrics import roc_curve, auc

def plot_roc_curves(task_results, task_name):
    """Plot ROC curves for the best fold of each model."""
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    colors = {'GRU': 'blue', 'LSTM': 'green', 'RF': 'red', 'XGBoost': 'orange'}

    for model_name, info in task_results['best_fold_info'].items():
        y_true = info['y_test']
        y_prob = info['y_prob']
        if y_prob is not None and len(np.unique(y_true)) > 1:
            fpr, tpr, _ = roc_curve(y_true, y_prob[:, 1])
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, color=colors[model_name],
                    label=f'{model_name} (AUC = {roc_auc:.4f}, Fold {info["fold_idx"]+1})')

    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.5)')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curves — {task_name} (Best Folds)')
    ax.legend(loc='lower right')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_roc_curves(detection_results, 'Detection')
plot_roc_curves(staging_results, 'Staging')

# ============================================================
# 4.6  Model Interpretability (XAI) — unchanged, kept for reference
# ============================================================

# [Your existing XAI code remains the same — no changes needed]
# ... (run_integrated_gradients, run_shap_analysis, run_lime_analysis)
# ... (XAI execution for Detection and Staging tasks)

# ============================================================
# FINAL: Save results to Drive (UPDATED with accuracy + tuning logs)
# ============================================================
SAVE_DIR = 'results/run1/'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save the results table
results_table.to_csv(os.path.join(SAVE_DIR, 'cv_results_summary.csv'), index=False)
print(f"\nResults saved to {SAVE_DIR}cv_results_summary.csv")

# Save detailed per-fold metrics (NOW with accuracy)
for task_name, task_data in all_results.items():
    for model_name, fold_metrics_list in task_data['fold_metrics'].items():
        rows = []
        for fold_idx, metrics in enumerate(fold_metrics_list):
            row = {
                'task': task_name,
                'model': model_name,
                'fold': fold_idx + 1,
                'accuracy': metrics['accuracy'],
                'precision': metrics['precision'],
                'recall': metrics['recall'],
                'f1_score': metrics['f1_score'],
                'auc_roc': metrics['auc_roc'],
            }
            if 'specificity' in metrics:
                row['specificity'] = metrics['specificity']
            rows.append(row)
        per_fold_df = pd.DataFrame(rows)
        fname = f'{task_name}_{model_name}_per_fold.csv'
        per_fold_df.to_csv(os.path.join(SAVE_DIR, fname), index=False)

# Save best hyperparameters per fold
for task_name, task_data in all_results.items():
    hp_rows = []
    for model_name, params_list in task_data['best_params_per_fold'].items():
        for fold_idx, params in enumerate(params_list):
            row = {'task': task_name, 'model': model_name, 'fold': fold_idx + 1}
            row.update(params)
            hp_rows.append(row)
    hp_df = pd.DataFrame(hp_rows)
    hp_fname = f'{task_name}_best_hyperparameters.csv'
    hp_df.to_csv(os.path.join(SAVE_DIR, hp_fname), index=False)
    print(f"  Saved: {hp_fname}")

# Save tuning logs (all configs tried per fold)
for task_name, task_data in all_results.items():
    for log_name, log_df in task_data['tuning_logs'].items():
        log_fname = f'{task_name}_{log_name}_tuning_log.csv'
        log_df.to_csv(os.path.join(SAVE_DIR, log_fname), index=False)

print("\nAll per-fold results, hyperparameters, and tuning logs saved.")
print("\n✅ EXPERIMENT COMPLETE (with Hyperparameter Tuning)!")

Using device: cuda
